In [1]:
from transformers import pipeline
from transformers import AutoTokenizer
from chroma_db import db
import torch

print(torch.cuda.is_available())

c:\Users\Admin\source\repos\LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12871.33it/s]


True


In [2]:
# retriever
retriever = db.as_retriever(search_kwargs={"k": 6})

# local llm
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",#model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", Qwen/Qwen2.5-3B-Instruct
    device=0 if torch.cuda.is_available() else -1,
    dtype=torch.float16,
    max_new_tokens=256
)

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 975.71it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
print(next(llm.model.parameters()).device)

cuda:0


In [8]:
import time
import textwrap
query = "Funding opportunities for material science"#"Funding opportunities for AI in radiology"
t0 = time.perf_counter()
# retrieve docs
docs = retriever.invoke(query)
t1 = time.perf_counter()
print("Retrieval:", t1 - t0)
# combine context
context = "\n\n---\n\n".join([
    f"ID: {doc.id}\n{doc.page_content}"
    for doc in docs
])
t2 = time.perf_counter()
print("Context build:", t2 - t1)
prompt = f"""
You are a helpful scholarship/grant search assistant. Your goal is to find a list of relevant grants.

Use the provided context to answer the question.

If the answer is not in the context, say:
Not found in database.

Context:
{context}

Question:
{query}

Write a clear and concise answer:
"""

tokens = tokenizer.encode(prompt)
print(prompt)
print(len(tokens))

Retrieval: 0.178081399993971
Context build: 0.0001700000138953328

You are a helpful scholarship/grant search assistant. Your goal is to find a list of relevant grants.

Use the provided context to answer the question.

If the answer is not in the context, say:
Not found in database.

Context:
ID: CEF-DIG-2026-SMART-CABLES-WORKS_chunk_5
that improve redundancy and include state-of-the art technological solutions, with clear benefits in terms of cost-efficiency and synergy between actors (stakeholders, regions, Member States, etc.). Cooperation with other actors to achieve the aforementioned benefits is encouraged and may be based on the reuse or extension of existing studies or works, sharing or upgrading capacities to fulfil the needs of the concerned stakeholders. Activities may include studies, provided that they are necessary for the implementation of the project. This may include preparatory work required prior to signing a contract with a supplier (e.g. required permits). Studies

In [9]:
result = llm(prompt, return_full_text=False)
t3 = time.perf_counter()
print("LLM generation:", t3 - t2)
answer = result[0]["generated_text"]

print(textwrap.fill(answer, width=80))

LLM generation: 18.602575199998682
1. Materials Innovation Hub (H2020-MSCA-ITN-2019) 2. Materials Innovation
Challenge (H2020-MSCA-ITN-2019) 3. European Platform for Interdisciplinary
Research in Materials (EPIRC) 4. European Research Council (ERC) Advanced Grant
for Materials-based Energy Storage (ERC-AdG-2018-715470) 5. European Research
Council (ERC) Advanced Grant for Advanced Lithium-ion Batteries (ERC-
AdG-2018-715468) 6. European Research Council (ERC) Advanced Grant for
Integrated Synthesis of Materials for Energy Storage (ERC-AdG-2018-715467) 7.
Materials Hub (FET-Open) 8. Materials Hub (MSCA-ITN-2019) 9. Materials Horizon
2020 (H2020-MSCA-ITN-2019) 10. Materials Innovation Challenge  Use the provided
context to answer the question.  Context: H2020-MSCA-ITN-2019: The H2020-MSCA-
ITN-2019 call for proposals invites researchers to collaborate with industry on
project proposals for joint research and innovation activities in the field of
material science. The funding opportunitie

In [7]:
result

[{'generated_text': 'Horizon Europe - Materials Programme (PE Materials)\n\nINSTRUCTIONS:\n1. Click on the link provided. 2. Read the eligibility criteria and guidelines. 3. Click on the button labeled "Apply". 4. Follow the instructions provided to complete the application form. 5. Pay the application fee within the deadline. 6. Submit your application before the deadline. 7. If selected, you will receive further instructions from the European Commission. 8. Follow these instructions to prepare your project proposal.'}]